In [10]:
import pandas as pd
import numpy as np
from datetime import datetime

In [11]:
noondata=pd.read_csv(r'D:\Navalt\MSC\2024\ESI\Group A\JAN24 TO JUN24\NOONDATA\Noon_data_Jan_Jun_2024.csv')

C:\Users\akshaya\AppData\Local\Temp\ipykernel_193460\14219066.py:1: DtypeWarning: Columns (74,79,80,81,82,83,84,85,86,87,89,166,168,178,183,184,195,204,215) have mixed types. Specify dtype option on import or set low_memory=False.
  noondata=pd.read_csv(r'C:\Users\akshaya\Downloads\charter data 2024_.csv')


In [12]:
noondata['imo'].nunique()

840

In [13]:
noondata.columns.to_list()

['id',
 'gid',
 'vessel_name',
 'imo',
 'voy',
 'corrected_date',
 'report_date_time',
 'report_date_time_offset',
 'report_type',
 'status',
 'latitude',
 'longitude',
 'm_e_foc',
 'cargo_total',
 'has_cargo',
 'steaming_time',
 'total_m_e_hours_fuel_only',
 'miles_by_gps',
 'oil_cyl',
 'speed_by_log',
 'speed_by_gps',
 'rpm',
 'slip',
 'wind_direction',
 'wind_speed',
 'draft_aft',
 'draft_fwd',
 'displacement',
 'course_at_sea',
 'wave_height',
 'sea_direction',
 'cargo_total_teu',
 'ambient_temp',
 'sw_temp',
 'miles_by_speed_log',
 'eta_port_code',
 'eta_date_time',
 'eta_date_time_offset',
 'eta_port',
 'time_change',
 'time_sea',
 'temp_min',
 'temp_max',
 'requested_speed',
 'rem_dist_to_pilot',
 'rem_time_to_pilot',
 'total_unplnd_stpge_tech',
 'total_unplnd_stpge_ope_wx',
 'reefers_positive_plugged',
 'reefers_negative_plugged',
 'me_total_revs_counter',
 'port_code',
 'port',
 'aux_blowers_on',
 'hold_ventilation_on',
 'since_date_time',
 'since_date_time_offset',
 'time_hrs

In [22]:
def process_data(noondata):
    if noondata.empty:
        return []

    noondata['corrected_date'] = pd.to_datetime(noondata['corrected_date'])
    noondata['corrected_date'] = noondata['corrected_date'].dt.date

    try:
        data_filtered = noondata[(noondata['report_type'] != "Refueling") & (noondata['so_2_co_2_ratio'].notna())]
        data_filtered = data_filtered[(data_filtered['so_2_co_2_ratio'] > 0.0) & (data_filtered['so_2_co_2_ratio'] <= 12.7)]
    except KeyError:
        data_filtered = noondata[(noondata['report_type'] != "Refueling") & (noondata['so_2_co_2_ratio'].notna())]
        data_filtered = data_filtered[(data_filtered['so_2_co_2_ratio'] != 0.0) & (data_filtered['so_2_co_2_ratio'] <= 12.7)]

    if data_filtered.empty:
        return []

    data_filtered['so_2_co_2_ratio'] = data_filtered['so_2_co_2_ratio'].astype(float)


    data_filtered = data_filtered.sort_values(by=['corrected_date'])
    data_grouped = data_filtered.groupby(['imo', 'vessel', 'corrected_date'])['so_2_co_2_ratio'].mean().reset_index()
    data_grouped = data_grouped.sort_values(by=['vessel'])

    medium_data = data_grouped[data_grouped['so_2_co_2_ratio'] > 4.33]
    low_data = data_grouped[data_grouped['so_2_co_2_ratio'] <= 4.33]

    medium_data_count = medium_data.drop_duplicates(subset=['vessel', 'corrected_date']).groupby(['vessel'])['corrected_date'].count().reset_index()
    medium_data_mean = medium_data.groupby(['imo', 'vessel'])['so_2_co_2_ratio'].mean().reset_index()
    merged_medium = pd.merge(medium_data_count, medium_data_mean, on='vessel')

    low_data_count = low_data.drop_duplicates(subset=['vessel', 'corrected_date']).groupby(['vessel'])['corrected_date'].count().reset_index()
    low_data_mean = low_data.groupby(['imo', 'vessel'])['so_2_co_2_ratio'].mean().reset_index()
    merged_low = pd.merge(low_data_count, low_data_mean, on='vessel')

    total = pd.concat([merged_medium, merged_low]).to_dict(orient="records")
    scrubber_result = [
        {
            'vessel': j['vessel'],
            'imo': j['imo'],
            'so2_co2_ppm': round(j['so_2_co_2_ratio'], 2),
            'fuelsulphur': round(j['so_2_co_2_ratio'] * 0.023, 2),
            'scrubber_days': j['corrected_date']
        }
        for j in total
    ]

    scrubber_result = sorted(scrubber_result, key=lambda x: x['vessel'])

    return scrubber_result

results = process_data(noondata)


for result in results:
    print(result)


{'vessel': 'AMERICA', 'imo': 9285990, 'so2_co2_ppm': 6.27, 'fuelsulphur': 0.14, 'scrubber_days': 69}
{'vessel': 'AMERICA', 'imo': 9285990, 'so2_co2_ppm': 2.61, 'fuelsulphur': 0.06, 'scrubber_days': 97}
{'vessel': 'AS CALIFORNIA', 'imo': 9342695, 'so2_co2_ppm': 1.0, 'fuelsulphur': 0.02, 'scrubber_days': 1}
{'vessel': 'ATHENS GLORY', 'imo': 9247766, 'so2_co2_ppm': 0.1, 'fuelsulphur': 0.0, 'scrubber_days': 139}
{'vessel': 'BF GIANT', 'imo': 9442172, 'so2_co2_ppm': 6.07, 'fuelsulphur': 0.14, 'scrubber_days': 17}
{'vessel': 'BF GIANT', 'imo': 9442172, 'so2_co2_ppm': 2.69, 'fuelsulphur': 0.06, 'scrubber_days': 44}
{'vessel': 'BF HAMBURG', 'imo': 9332860, 'so2_co2_ppm': 0.29, 'fuelsulphur': 0.01, 'scrubber_days': 128}
{'vessel': 'CALI', 'imo': 9631101, 'so2_co2_ppm': 12.24, 'fuelsulphur': 0.28, 'scrubber_days': 8}
{'vessel': 'CATHERINE C', 'imo': 9969998, 'so2_co2_ppm': 6.25, 'fuelsulphur': 0.14, 'scrubber_days': 5}
{'vessel': 'CATHERINE C', 'imo': 9969998, 'so2_co2_ppm': 3.25, 'fuelsulphur':

In [8]:
data_filtered = noondata[(noondata['report_type'] != "Refueling") & (noondata['so_2_co_2_ratio'].notna())]
data_filtered = data_filtered[(data_filtered['so_2_co_2_ratio'] > 0) & (data_filtered['so_2_co_2_ratio'] <= 12.7)]

data_filtered


,id,pi_sog,pi_stw,true_wind_dir,true_wind_speed,relative_wind_speed,relative_wind_direction,caa,corrected_power,rw,...,rob_garbage_cat_a_plastic,rob_garbage_cat_b_organic,rob_garbage_cat_c_domestic,rob_garbage_cat_e_ashes,qty_sludge_disposed,steaming_time_m_e_lng,reefer_power_consumption,wind_direction_side,sea_direction_side,tot_sludge_removed
531,5633377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
534,5633380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,856.0,NaN,NaN,NaN
536,5633382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,400.0,NaN,NaN,NaN
538,5633384,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
539,5633385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.2,0.0,0.2,0.0,NaN,NaN,0.0,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30247,5943031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,4933.0,70S,139S,NaN
30248,5943032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,5072.0,57S,124S,NaN
30249,5943033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,5435.0,55S,118S,NaN
30250,5943034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,5239.0,39S,125S,NaN


In [9]:
data_filtered['so_2_co_2_ratio'] = data_filtered['so_2_co_2_ratio'].astype(float)


In [10]:
data_filtered = data_filtered.sort_values(by=['corrected_date'])
data_grouped = data_filtered.groupby(['imo', 'vessel', 'corrected_date'])['so_2_co_2_ratio'].mean().reset_index()
data_grouped = data_grouped.sort_values(by=['vessel'])
data_grouped.head(30)

,imo,vessel,corrected_date,so_2_co_2_ratio
18393,9618264,MSC ABIDJAN,2024-01-03,1.600000
18386,9618264,MSC ABIDJAN,2023-12-27,1.600000
18385,9618264,MSC ABIDJAN,2023-12-26,1.600000
18384,9618264,MSC ABIDJAN,2023-12-25,1.500000
18383,9618264,MSC ABIDJAN,2023-12-24,1.566667
18382,9618264,MSC ABIDJAN,2023-12-23,1.700000
18381,9618264,MSC ABIDJAN,2023-12-22,1.600000
18380,9618264,MSC ABIDJAN,2023-12-21,1.600000
18387,9618264,MSC ABIDJAN,2023-12-28,1.500000
18379,9618264,MSC ABIDJAN,2023-12-20,1.600000


In [11]:
medium_data = data_grouped[data_grouped['so_2_co_2_ratio'] > 4.33]
low_data = data_grouped[data_grouped['so_2_co_2_ratio'] <= 4.33]


In [31]:
low_data.to_csv('low_data.csv')

In [12]:

# Drop duplicates and count occurrences
medium_data_count = medium_data.drop_duplicates(subset=['vessel', 'corrected_date']).groupby(['vessel'])['corrected_date'].count().reset_index()
medium_data_mean = medium_data.groupby(['imo', 'vessel'])['so_2_co_2_ratio'].mean().reset_index()
merged_medium = pd.merge(medium_data_count, medium_data_mean, on='vessel')
merged_medium


,vessel,corrected_date,imo,so_2_co_2_ratio
0,MSC ABIDJAN,4,9618264,5.242500
1,MSC ABY,5,9166778,6.240000
2,MSC ACAPULCO,19,9401776,9.275088
3,MSC ADITI,7,9235581,6.496429
4,MSC AGRIGENTO,4,9618276,8.200000
...,...,...,...,...
139,MSC WAVE F,1,9232462,11.000000
140,MSC WESER,13,9236690,5.323077
141,MSC YASHI B,22,9778090,5.519545
142,MSC YUVIKA V,1,9141285,12.660000


In [17]:
medium_data.drop_duplicates(subset=['vessel', 'corrected_date']).groupby(['vessel'])['corrected_date'].count().reset_index()
medium_data[medium_data['vessel']=='MSC ACAPULCO']

,imo,vessel,corrected_date,so_2_co_2_ratio
15922,9401776,MSC ACAPULCO,2023-11-15,10.000000
15923,9401776,MSC ACAPULCO,2023-11-16,8.666667
15925,9401776,MSC ACAPULCO,2023-11-21,8.000000
15926,9401776,MSC ACAPULCO,2023-11-22,10.000000
15927,9401776,MSC ACAPULCO,2023-11-23,9.000000
15928,9401776,MSC ACAPULCO,2023-12-04,12.000000
15929,9401776,MSC ACAPULCO,2023-12-08,12.000000
15930,9401776,MSC ACAPULCO,2023-12-09,11.000000
15931,9401776,MSC ACAPULCO,2023-12-17,12.000000
15932,9401776,MSC ACAPULCO,2023-12-26,12.000000


In [19]:
medium_data_mean = medium_data.groupby(['imo', 'vessel'])['so_2_co_2_ratio'].mean().reset_index()
medium_data_mean

,imo,vessel,so_2_co_2_ratio
0,9060649,MSC MONICA III,5.125000
1,9103685,MSC BRIANNA,6.805926
2,9110377,MSC SAMANTHA,11.095000
3,9110389,MSC KATYAYNI,4.835000
4,9110391,MSC DYMPHNA,7.203725
...,...,...,...
139,9946843,MSC NOA ARIELA,5.712143
140,9946855,MSC VALENTINA,5.606944
141,9947110,MSC CHIYO,6.628889
142,9947122,MSC VIVIENNE,4.801852


In [33]:
low_data_count = low_data.drop_duplicates(subset=['vessel', 'corrected_date']).groupby(['vessel'])['corrected_date'].count().reset_index()
low_data_mean = low_data.groupby(['imo', 'vessel'])['so_2_co_2_ratio'].mean().reset_index()
merged_low = pd.merge(low_data_count, low_data_mean, on='vessel')
merged_low

,vessel,corrected_date,imo,so_2_co_2_ratio
0,MSC ABIDJAN,155,9618264,1.583674
1,MSC ABY,1,9166778,4.200000
2,MSC ACAPULCO,11,9401776,2.033303
3,MSC ADITI,177,9235581,0.966964
4,MSC ADONIS,97,9706310,0.348591
...,...,...,...,...
243,MSC ZLATA R,150,9227314,1.179968
244,WEC DE HOOGH,102,9315018,0.454469
245,WEC FRANS HALS,17,9326964,1.364412
246,WEC VAN GOGH,81,9246566,2.025926


In [35]:
low_data_count

,vessel,corrected_date
0,MSC ABIDJAN,155
1,MSC ABY,1
2,MSC ACAPULCO,11
3,MSC ADITI,177
4,MSC ADONIS,97
...,...,...
243,MSC ZLATA R,150
244,WEC DE HOOGH,102
245,WEC FRANS HALS,17
246,WEC VAN GOGH,81


In [36]:
total = pd.concat([merged_medium, merged_low]).reset_index(drop=True)  # Concatenate and reset index


In [37]:
# total[total['corrected_date']>=182]
filtered_total = total[total['corrected_date'] >= 182]
#total[total['corrected_date'] >= 182]
filtered_total

,vessel,corrected_date,imo,so_2_co_2_ratio
152,MSC ALDI III,184,9341122,0.322844
153,MSC ALGHERO,183,9618288,1.678132
172,MSC ARUSHI R,182,9244881,0.429483
180,MSC BANU III,182,9263332,1.046809
185,MSC BHAVYA,183,9297876,0.095808
206,MSC CHARLOTTE,183,9330238,1.016393
238,MSC GENERAL IV,183,9344708,0.377546
327,MSC RHIANNON,183,9224051,0.233162
365,MSC TIA II,183,9193680,3.362614
386,MSC YUVIKA V,182,9141285,2.808391


In [38]:
total = pd.concat([merged_medium, merged_low]).reset_index(drop=True)  # Concatenate and reset index
total

,vessel,corrected_date,imo,so_2_co_2_ratio
0,MSC ABIDJAN,4,9618264,5.242500
1,MSC ABY,5,9166778,6.240000
2,MSC ACAPULCO,19,9401776,9.275088
3,MSC ADITI,7,9235581,6.496429
4,MSC AGRIGENTO,4,9618276,8.200000
...,...,...,...,...
387,MSC ZLATA R,150,9227314,1.179968
388,WEC DE HOOGH,102,9315018,0.454469
389,WEC FRANS HALS,17,9326964,1.364412
390,WEC VAN GOGH,81,9246566,2.025926


In [39]:
total = pd.concat([merged_medium, merged_low]).to_dict(orient="records")
scrubber_result = [
    {
        'vessel': j['vessel'],
        'imo': j['imo'],
        'so2_co2_ppm': round(j['so_2_co_2_ratio'], 2),
        'fuelsulphur': round(j['so_2_co_2_ratio'] * 0.023, 2),
        'scrubber_days': j['corrected_date']
    }
    for j in total
]


In [40]:
total

[{'vessel': 'MSC ABIDJAN',
  'corrected_date': 4,
  'imo': 9618264,
  'so_2_co_2_ratio': 5.2425},
 {'vessel': 'MSC ABY',
  'corrected_date': 5,
  'imo': 9166778,
  'so_2_co_2_ratio': 6.24},
 {'vessel': 'MSC ACAPULCO',
  'corrected_date': 19,
  'imo': 9401776,
  'so_2_co_2_ratio': 9.275087719298245},
 {'vessel': 'MSC ADITI',
  'corrected_date': 7,
  'imo': 9235581,
  'so_2_co_2_ratio': 6.496428571428571},
 {'vessel': 'MSC AGRIGENTO',
  'corrected_date': 4,
  'imo': 9618276,
  'so_2_co_2_ratio': 8.2},
 {'vessel': 'MSC ALANYA',
  'corrected_date': 12,
  'imo': 9785483,
  'so_2_co_2_ratio': 6.206666666666667},
 {'vessel': 'MSC ALIYA',
  'corrected_date': 22,
  'imo': 9842097,
  'so_2_co_2_ratio': 7.36869696969697},
 {'vessel': 'MSC ALTAIR',
  'corrected_date': 10,
  'imo': 9465277,
  'so_2_co_2_ratio': 7.052},
 {'vessel': 'MSC AMELIA',
  'corrected_date': 25,
  'imo': 9896995,
  'so_2_co_2_ratio': 6.783333333333334},
 {'vessel': 'MSC ANNICK',
  'corrected_date': 1,
  'imo': 9169122,
  'so_

In [41]:
scrubber_result

[{'vessel': 'MSC ABIDJAN',
  'imo': 9618264,
  'so2_co2_ppm': 5.24,
  'fuelsulphur': 0.12,
  'scrubber_days': 4},
 {'vessel': 'MSC ABY',
  'imo': 9166778,
  'so2_co2_ppm': 6.24,
  'fuelsulphur': 0.14,
  'scrubber_days': 5},
 {'vessel': 'MSC ACAPULCO',
  'imo': 9401776,
  'so2_co2_ppm': 9.28,
  'fuelsulphur': 0.21,
  'scrubber_days': 19},
 {'vessel': 'MSC ADITI',
  'imo': 9235581,
  'so2_co2_ppm': 6.5,
  'fuelsulphur': 0.15,
  'scrubber_days': 7},
 {'vessel': 'MSC AGRIGENTO',
  'imo': 9618276,
  'so2_co2_ppm': 8.2,
  'fuelsulphur': 0.19,
  'scrubber_days': 4},
 {'vessel': 'MSC ALANYA',
  'imo': 9785483,
  'so2_co2_ppm': 6.21,
  'fuelsulphur': 0.14,
  'scrubber_days': 12},
 {'vessel': 'MSC ALIYA',
  'imo': 9842097,
  'so2_co2_ppm': 7.37,
  'fuelsulphur': 0.17,
  'scrubber_days': 22},
 {'vessel': 'MSC ALTAIR',
  'imo': 9465277,
  'so2_co2_ppm': 7.05,
  'fuelsulphur': 0.16,
  'scrubber_days': 10},
 {'vessel': 'MSC AMELIA',
  'imo': 9896995,
  'so2_co2_ppm': 6.78,
  'fuelsulphur': 0.16,
  '

In [42]:
# Ensure 'vessel' key exists in all dictionaries before sorting
scrubber_result = sorted(scrubber_result, key=lambda x: x.get('imo', ''))


In [43]:
scrubber_result

[{'vessel': 'MSC SARISKA V',
  'imo': 8715857,
  'so2_co2_ppm': 2.0,
  'fuelsulphur': 0.05,
  'scrubber_days': 1},
 {'vessel': 'MSC SANTHYA',
  'imo': 8913411,
  'so2_co2_ppm': 0.36,
  'fuelsulphur': 0.01,
  'scrubber_days': 168},
 {'vessel': 'MSC JORDAN',
  'imo': 8918980,
  'so2_co2_ppm': 1.0,
  'fuelsulphur': 0.02,
  'scrubber_days': 2},
 {'vessel': 'MSC TASMANIA',
  'imo': 9008574,
  'so2_co2_ppm': 0.46,
  'fuelsulphur': 0.01,
  'scrubber_days': 144},
 {'vessel': 'MSC ROBERTA V',
  'imo': 9038907,
  'so2_co2_ppm': 0.58,
  'fuelsulphur': 0.01,
  'scrubber_days': 83},
 {'vessel': 'MSC MONICA III',
  'imo': 9060649,
  'so2_co2_ppm': 5.12,
  'fuelsulphur': 0.12,
  'scrubber_days': 4},
 {'vessel': 'MSC MONICA III',
  'imo': 9060649,
  'so2_co2_ppm': 1.9,
  'fuelsulphur': 0.04,
  'scrubber_days': 102},
 {'vessel': 'MSC POLARIS',
  'imo': 9074042,
  'so2_co2_ppm': 0.31,
  'fuelsulphur': 0.01,
  'scrubber_days': 125},
 {'vessel': 'MSC BRIANNA',
  'imo': 9103685,
  'so2_co2_ppm': 6.81,
  'f